In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/"

'Colab Notebooks'


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib
import os

# caricamento dati
df = pd.read_csv("/content/drive/MyDrive/prototipo_oliveto_pw/data/ticket_museo_le_nuove.csv")

df["text"] = df["subject"] + " " + df["description"]

Listing contents of /content/drive/MyDrive/prototipo_oliveto_pw/data/
ls: cannot access '/content/drive/MyDrive/prototipo_oliveto_pw/data/': No such file or directory


In [ ]:
# feature e target
category_map = {
    "Amministrazione": 0,
    "Tecnico": 1,
    "Commerciale": 2
}

y = df["category"].map(category_map).values

vectorizer = TfidfVectorizer(max_features=2000)
X = vectorizer.fit_transform(df["text"]).toarray()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [ ]:
# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#modello di rete neurale
class TicketClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

# instanzia il modello:
#   X.shape[1] corrisponde al numero di ticket
#   3 è il numero di classi target
model_cat = TicketClassifier(X.shape[1], 3)
# definisce la loss function per la classificazione multiclasse
criterion = nn.CrossEntropyLoss()
# l'ottimizzatore usa Adam, definendo di andare a ad aggiornare model_cat
#   con un learning rate di 0.001 più stabile di 0.01
optimizer = torch.optim.Adam(model_cat.parameters(), lr=0.001)

In [ ]:
#ciclo di addestramento
for epoch in range(21):
    # pulisce i gradienti calcolati all'epoca precedente
    optimizer.zero_grad()
    # forward pass (predizione)
    outputs = model_cat(X_train)
    # calcolo della loss
    loss = criterion(outputs, y_train)
    # back-propagation
    loss.backward()
    # aggiornamento dei pesi
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/prototipo_oliveto_pw/models"
os.makedirs(MODEL_DIR, exist_ok=True)

torch.save(
    model_cat.state_dict(),
    MODEL_DIR + "/category_model.pt"
)

joblib.dump(
    vectorizer,
    MODEL_DIR + "/tfidf_vectorizer.joblib"
)

print("✅ category_model.pt e TF-IDF salvati")
